> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 4 · Notebook 01 — Environment, config and secrets

**Sessions:** S1 (Reproducible environment) · S2 (IB architecture) · [Lesson plan](../../docs/lessons/PART_04_BROKER_CONNECTIVITY.md) · graded labs in [`labs/part04/`](../../labs/part04/)

**You will:**
1. Load settings from environment variables and keep secrets out of printouts.
2. Map IB Gateway / TWS ports and refuse a live port in a paper environment.
3. Write the secret scanner a pre-commit hook runs before every commit.
4. See why every process needs its own IB `clientId`.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing here connects to a broker: the account rows, bars, ticks and order events are synthetic, shaped like what `ib_async` and `alpaca-py` return.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p4lib.py is in notebooks/part04/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p4lib as p

p.use_course_style()

## 1. Settings come from the environment, never from the code

In the project, `pydantic-settings` reads `QF_*` variables from `.env`. Here we pass a dict to the same kind of model. `SecretStr` keeps the key out of `print`, logs and tracebacks; you ask for it explicitly with `.get_secret_value()`.

In [ ]:
env = {"QF_ENV": "paper", "QF_IB_PORT": "4002", "QF_IB_CLIENT_ID": "11",
       "QF_ALPACA_KEY": "paper-key-id", "QF_ALPACA_SECRET": "paper-secret-value", "HOME": "/home/me"}
settings = p.settings_from_env(env)
print(settings)                                   # the secrets print as '**********'
print(repr(settings.alpaca_secret))
print("ib_port is an int:", type(settings.ib_port).__name__, settings.ib_port)
print("the real value, only when you ask:", settings.alpaca_secret.get_secret_value()[:6] + "…")

## 2. IB ports: which door are you knocking on?

| App | Paper | Live |
|---|---|---|
| IB Gateway | 4002 | 4001 |
| TWS | 7497 | 7496 |

The port is the only thing between a notebook experiment and a real-money order. Know them by heart.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
PORTS = {("gateway", "paper"): 4002, ("gateway", "live"): 4001,
         ("tws", "paper"): 7497, ("tws", "live"): 7496}

def ib_port(app: str, mode: str) -> int:
    return PORTS[(app, mode)]

combos = [(a, m) for a in ("gateway", "tws") for m in ("paper", "live")]
mine = [ib_port(a, m) for a, m in combos]
mine = p.check("IB ports", mine, [p.ib_port(a, m) for a, m in combos])
dict(zip(combos, mine))

Now the guard that runs at startup. A paper (or backtest) environment must **never** reach a live port. A live environment on a paper port is also a mistake: you'd think you were trading and you're not.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def port_problem(env: str, port: int) -> str | None:
    live, paper = {4001, 7496}, {4002, 7497}
    if port not in live | paper:
        return "unknown port"
    if env in ("paper", "backtest") and port in live:
        return "live port in a paper environment"
    if env == "live" and port in paper:
        return "paper port in a live environment"
    return None

cases = [("paper", 4002), ("paper", 4001), ("backtest", 7496), ("live", 7497), ("live", 4001), ("paper", 5000)]
mine = [port_problem(e, port) for e, port in cases]
mine = p.check("port guard", mine, [p.port_problem(e, port) for e, port in cases])
list(zip(cases, mine))

In [ ]:
def connect(settings):
    problem = p.port_problem(settings.env, settings.ib_port)
    if problem:
        raise RuntimeError(f"refusing to connect: {problem}")
    return f"would connect to {settings.ib_host}:{settings.ib_port} as clientId {settings.ib_client_id}"

print(connect(settings))
try:
    connect(p.settings_from_env({**env, "QF_IB_PORT": "4001"}))
except RuntimeError as e:
    print("🛑", e)

## 3. A secret scanner for pre-commit

Leaked keys are the most common real incident in retail algo trading. The scanner flags two things:
* an **Alpaca key id**: `PK` or `AK` followed by 18 capitals/digits (`p.ALPACA_KEY_ID`);
* an **assignment** such as `secret = ...` / `api_key: ...` with a value of 8+ characters (`p.ASSIGNMENT`), unless the value is a placeholder: it starts with `<`, `$` or `{`, or is `changeme`.

The regex group 1 of `ASSIGNMENT` is the value.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def find_secrets(text: str) -> list[tuple[int, str]]:
    hits = []
    for n, line in enumerate(text.splitlines(), start=1):
        if p.ALPACA_KEY_ID.search(line):
            hits.append((n, "alpaca_key_id"))
            continue
        m = p.ASSIGNMENT.search(line)
        if m and not (m.group(1)[0] in "<${" or m.group(1).lower() == "changeme"):
            hits.append((n, "assignment"))
    return hits

for name, text in p.SAMPLE_FILES.items():
    print(f"--- {name}")
    print(text)
mine = {name: find_secrets(text) for name, text in p.SAMPLE_FILES.items()}
mine = p.check("find_secrets", mine, {name: p.find_secrets(text) for name, text in p.SAMPLE_FILES.items()})
mine

Notice what the scanner **misses**: `api_key = 'abc'` is too short to flag, and a key split over two lines passes. Regex scanners catch the common mistake; they are not a guarantee. The real defence is that keys only ever live in `.env` (git-ignored) or a secret store.

## 4. One `clientId` per process

IB allows several API connections to one Gateway, but each needs a different `clientId`. A second connection with an id already in use gets **error 326** and is dropped, often mid-session. Allocate ids in config, per process, and assert at startup.

In [ ]:
from collections import Counter

processes = {"strategy_momentum": 11, "downloader": 12, "monitor": 13, "notebook": 11}
clash = {cid: [name for name, c in processes.items() if c == cid]
         for cid, n in Counter(processes.values()).items() if n > 1}
print("clientId clashes:", clash or "none")
print("→ the second of", clash.get(11), "to connect gets error 326 and loses its connection")

## Wrap-up

* Settings from the environment, secrets as `SecretStr`, and `.env` in `.gitignore`.
* The port is the paper/live switch: guard it at startup.
* A pre-commit secret scan catches the classic leak before it reaches Git history.
* Next: the graded version is `labs/part04/week13_foundations` (`ib_port`, `check_port_matches_env`, `find_secrets`).